# Course Registration Portal
### 1. Constants and File Handling
Reads and saves student, course, and registration records to `.txt` files

In [1]:
STUDENT_FILE = "students.txt"
COURSE_FILE = "courses.txt"
REG_FILE = "registrations.txt"
MAX_CREDITS = 18

def read_lines(filename):
    lines = []
    file = None
    try:
        file = open(filename, "r")
        for line in file:
            line = line.strip()
            if line != "":
                lines.append(line.split(","))
    except FileNotFoundError:
        print(filename, "not found. Starting with empty data.")
    except PermissionError:
        print("No permission to read", filename)
    finally:
        if file != None:
            file.close()
    return lines

def save_all(students, courses, registrations):
    try:
        file = open(STUDENT_FILE, "w")
        for sid in students:
            file.write(sid + "," + students[sid]["name"] + "," + students[sid]["password"] + "\n")
        file.close()

        file = open(COURSE_FILE, "w")
        for code in courses:
            info = courses[code]
            file.write(code + "," + info[0] + "," + str(info[1]) + "," + str(info[2]) + "\n")
        file.close()

        file = open(REG_FILE, "w")
        for sid in registrations:
            for code in registrations[sid]:
                file.write(sid + "," + code + "\n")
        file.close()
    except PermissionError:
        print("Cannot save: no permission to write the files.")

def load_data():
    students = {}
    courses = {}
    registrations = {}

    try:
        for row in read_lines(STUDENT_FILE):
            students[row[0]] = {"name": row[1], "password": row[2]}

        for row in read_lines(COURSE_FILE):
            courses[row[0]] = (row[1], int(row[2]), int(row[3]))

        for row in read_lines(REG_FILE):
            sid = row[0]
            code = row[1]
            if sid not in registrations:
                registrations[sid] = set()
            registrations[sid].add(code)
    except ValueError:
        print("A file has a number in the wrong format.")
    except IndexError:
        print("A file has a broken line.")

    return students, courses, registrations

def make_sample_data(students, courses):
    
    if len(courses) == 0:
        courses["CS101"] = ("Programming Fundamentals", 3, 3)
        courses["MG201"] = ("Principles of Management", 3, 30)
        courses["ST301"] = ("Statistics for Analytics", 3, 30)
        courses["MK210"] = ("Marketing Basics", 2, 25)
    if len(students) == 0:
        students["S001"] = {"name": "Ayesha Khan", "password": "pass123"}
        students["S002"] = {"name": "Bilal Ahmed", "password": "pass456"}

### 2. Helper and Login Functions
Checks seat counts, calculates total credits, and verifies student credentials

In [2]:
def count_seats_used(code, registrations):
    count = 0
    for sid in registrations:
        if code in registrations[sid]:
            count += 1
    return count

def total_credits(sid, courses, registrations):
    total = 0
    if sid in registrations:
        for code in registrations[sid]:
            total += courses[code][1]
    return total

def student_login(username, password, students):
    if username in students and students[username]["password"] == password:
        return True
    return False

### 3. Course Operations
Handles course searching, registration (with seat and credit checks), dropping, and viewing.

In [3]:
def search_courses(courses, text):
    results = []
    text = text.lower()
    for code in courses:
        if text in courses[code][0].lower():
            results.append(code + " - " + courses[code][0])
    return results

def register_course(sid, code, students, courses, registrations):
    if sid not in students:
        return "Error: student not found."
    if code not in courses:
        return "Error: course " + code + " does not exist."
    if sid in registrations and code in registrations[sid]:
        return "Error: already registered in " + code + "."
    if count_seats_used(code, registrations) >= courses[code][2]:
        return "Error: " + code + " is full."

    new_total = total_credits(sid, courses, registrations) + courses[code][1]
    if new_total > MAX_CREDITS:
        return "Error: that would be " + str(new_total) + " credits. Limit is " + str(MAX_CREDITS) + "."

    if sid not in registrations:
        registrations[sid] = set()
    registrations[sid].add(code)
    save_all(students, courses, registrations)
    return "Success: registered in " + code + " - " + courses[code][0] + "."

def drop_course(sid, code, students, courses, registrations):
    if sid in registrations and code in registrations[sid]:
        registrations[sid].remove(code)
        save_all(students, courses, registrations)
        return "Course " + code + " dropped."
    return "Error: you are not registered in " + code + "."

def show_courses(courses, registrations):
    print("\nCODE    TITLE                         CREDITS  SEATS LEFT")
    for code in courses:
        info = courses[code]
        left = info[2] - count_seats_used(code, registrations)
        print(code, "|", info[0], "|", info[1], "credits |", left, "seats left")

### 4. Menus and Main Program
Runs the student menu and main portal loop

In [4]:
def student_menu(sid, students, courses, registrations):
    print("\nWelcome,", students[sid]["name"])
    while True:
        print("\nSTUDENT MENU ")
        print("1. View courses")
        print("2. Search courses")
        print("3. Register in a course")
        print("4. Drop a course")
        print("5. My courses")
        print("6. Logout")
        choice = input("Choose: ")

        if choice == "1":
            show_courses(courses, registrations)
        elif choice == "2":
            text = input("Search text: ")
            results = search_courses(courses, text)
            for item in results:
                print(item)
            if len(results) == 0:
                print("Nothing found.")
        elif choice == "3":
            code = input("Course code: ")
            print(register_course(sid, code, students, courses, registrations))
        elif choice == "4":
            code = input("Course code to drop: ")
            print(drop_course(sid, code, students, courses, registrations))
        elif choice == "5":
            if sid in registrations:
                for code in registrations[sid]:
                    print(code, "-", courses[code][0])
            print("Total credits:", total_credits(sid, courses, registrations))
        elif choice == "6":
            print("Logged out.")
            break
        else:
            print("Please type a number from 1 to 6.")

def main():
    students, courses, registrations = load_data()
    make_sample_data(students, courses)
    save_all(students, courses, registrations)
    print("Loaded", len(students), "students and", len(courses), "courses.")

    while True:
        print("\nCOURSE REGISTRATION PORTAL")
        print("1. Student login")
        print("2. Exit")
        choice = input("Choose: ")

        if choice == "2":
            print("Goodbye!")
            break
        elif choice == "1":
            username = input("Student ID: ")
            password = input("Password: ")
            if student_login(username, password, students):
                student_menu(username, students, courses, registrations)
            else:
                print("Wrong ID or password.")
        else:
            print("Please type 1 or 2.")
main()

Loaded 2 students and 4 courses.

COURSE REGISTRATION PORTAL
1. Student login
2. Exit


Choose:  1
Student ID:  S002
Password:  pass456



Welcome, Bilal Ahmed

STUDENT MENU 
1. View courses
2. Search courses
3. Register in a course
4. Drop a course
5. My courses
6. Logout


Choose:  1



CODE    TITLE                         CREDITS  SEATS LEFT
CS101 | Programming Fundamentals | 3 credits | 1 seats left
MG201 | Principles of Management | 3 credits | 30 seats left
ST301 | Statistics for Analytics | 3 credits | 30 seats left
MK210 | Marketing Basics | 2 credits | 25 seats left

STUDENT MENU 
1. View courses
2. Search courses
3. Register in a course
4. Drop a course
5. My courses
6. Logout


Choose:  2
Search text:  Marketing Basics


... (9 outputs left out)

Choose:  6


Logged out.

COURSE REGISTRATION PORTAL
1. Student login
2. Exit


Choose:  2


Goodbye!
